# Complexity-04b — La semaine historique de l'algorithmique online : secrétaire matroïdal et k-server

**Navigation** : [Index](README.md) | [Version Python](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) | [04 — Décider sans connaître la suite](Complexity-04-OnlineAlgorithms-CSharp.ipynb) | [05 — Compter est plus dur que vérifier](Complexity-05-CountingHarder-Permanent-CSharp.ipynb)

**Approfondissement de [04 — Décider sans connaître la suite](Complexity-04-OnlineAlgorithms-CSharp.ipynb)** | [Index de la série](README.md)

> **Public : Recherche.** Prérequis C# : tableaux, fonctions locales, dictionnaires, `Func<>` et requêtes `LINQ` de base ; exécution avec le noyau .NET Interactive (`.net-csharp`). Aucune bibliothèque externe : tout est écrit contre la BCL, y compris le générateur aléatoire et les structures du matroïde graphique. Le carnet suppose le modèle en ligne de 04 (décision irréversible, rapport de compétitivité) et ses trois objets classiques ; il leur ajoute la structure de **matroïde** et son oracle d'indépendance, la **work function** et les métriques finies.

**Ce que ce carnet suppose en plus de sa base** : la baseline $1/e$ désormais posée en 04 (§4), dont la section 1 reprend la mesure, et rien d'autre — les sections 2 à 4 se lisent avec le seul bagage du carnet 04.

> **Hommage** : Sahil Singla (*The Matroid Secretary Conjecture is True*, arXiv:2609.14555, 13/09/2026) et Christian Coester, Elias Koutsoupias, Marek Zbysiński (*The k-server conjecture is true*, arXiv:2609.15979, 14/09/2026). Les deux PDF sont archivés depuis le **16/09/2026** dans le gisement de bibliographie partagé ; chaque renvoi de ce carnet donne la **page** du PDF archivé, identique à celle citée par la version Python.

Septembre 2026 : **deux conjectures fondatrices de l'algorithmique online tombent à environ vingt-quatre heures d'écart**, par deux équipes différentes — la conjecture du **secrétaire matroïdal** (Babaioff–Immorlica–Kleinberg, 2007) et celle du **k-server** (Manasse–McGeoch–Sleator–Tarjan, ~1990). La saisie pédagogique du dépôt se fait par trois carnets liés : ce binôme synthétique côté Complexity, et deux distillations sœurs — `Complexity-04c-KServer-WorkFunction` (k-server : WFA exact sur métriques jouets) et `Complexity-04d-Secretaire-Matroidal` (l'algorithme de Singla lui-même, que ce carnet cite sans reproduire).

Ce carnet est le **jumeau C#** du carnet Python homonyme. La règle de la série est conservée : **faire tourner** les objets — simulations Monte Carlo, implémentation exacte du work function algorithm sur petites métriques — mesurer ce qui se mesure, et **citer sans reproduire** les preuves. Les budgets d'expérience de la version Python sont conservés à l'identique : mêmes nombres de tirages, mêmes huit graines, mêmes longueurs de séquence. Le générateur, lui, change de famille — voir la cellule suivante, qui dit exactement ce que cela déplace.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

Console.WriteLine($"Runtime .NET {Environment.Version}; culture du noyau : {CultureInfo.CurrentCulture.Name}");
Console.WriteLine("Les nombres s'affichent avec le separateur decimal de cette culture -- la virgule ici ; aucun calcul n'en depend.");

// Le generateur est celui de la BCL, seme une seule fois pour tout le carnet.
// La version Python tire ses permutations de PCG64 (numpy default_rng) : les deux
// suites ne coincident pas terme a terme. Ce que le portage conserve, c'est le
// protocole — memes budgets, memes graines, memes experiences — et les grandeurs
// theoriques comparees (1/e, borne k, ratio du WFA). Les valeurs tirees, elles,
// sont propres a ce carnet et sont affichees telles quelles.
var rng = new Random(42);

// Permutation de 0..n-1 par Fisher-Yates : c'est l'ordre d'arrivee aleatoire
// du modele en ligne. Elle est parametree par le generateur pour que les
// controles d'exercice puissent la rejouer a graine fixe.
int[] Permutation(int n, Random r)
{
    var p = Enumerable.Range(0, n).ToArray();
    for (int i = n - 1; i > 0; i--)
    {
        int j = r.Next(i + 1);
        (p[i], p[j]) = (p[j], p[i]);
    }
    return p;
}

Console.WriteLine("Generateur : System.Random (BCL), graine 42 ; permutations par Fisher-Yates.");

Runtime .NET 10.0.13; culture du noyau : fr-FR


Les nombres s'affichent avec le separateur decimal de cette culture -- la virgule ici ; aucun calcul n'en depend.


Generateur : System.Random (BCL), graine 42 ; permutations par Fisher-Yates.


## 1. Le secrétaire classique — la baseline à battre

**Le problème** (garder la formulation historique) : $n$ candidates et candidats se présentent dans un ordre aléatoire ; on ne connaît que leur **rang relatif** aux personnes déjà vues ; toute acceptation est **irréversible**. Objectif : maximiser la probabilité d'accepter la meilleure personne.

**L'algorithme $1/e$** (la règle des 37 %) : on observe sans accepter les $\lfloor n/e \rfloor$ premier·e·s arrivant·e·s, puis on accepte la première personne qui surpasse toutes les observées. La probabilité de capturer le meilleur tend vers $1/e \approx 0{,}368$ — c'est **la** référence que toute la littérature du secrétaire matroïdal cite comme point de départ.

En C#, la permutation se tire par Fisher-Yates sur un `Random` semé, et la comparaison « surpasse toutes les observées » se fait sur les **rangs** : la valeur du rang $n-1$ est le meilleur candidat par construction, donc la fonction n'a qu'un entier à retenir.

In [2]:
// Renvoie true si la regle 1/e selectionne le meilleur rang (n-1) de la permutation.
bool SecretaireClassique(int n, int[] perm)
{
    int seuil = Math.Max(1, (int)(n / Math.E));   // on ecarte les floor(n/e) premiers arrives
    int meilleurVu = -1;
    for (int i = 0; i < seuil; i++) meilleurVu = Math.Max(meilleurVu, perm[i]);
    for (int i = seuil; i < n; i++)
        if (perm[i] > meilleurVu) return perm[i] == n - 1;   // premier record : on accepte
    return false;
}

double MonteCarloClassique(int n, int runs, Random r)
{
    int succes = 0;
    for (int k = 0; k < runs; k++) if (SecretaireClassique(n, Permutation(n, r))) succes++;
    return (double)succes / runs;
}

foreach (int n in new[] { 20, 100, 1000 })
{
    double p = MonteCarloClassique(n, 20_000, rng);
    Console.WriteLine($"n={n,5}  P(meilleur selectionne) = {p:F4}   (theorie : 1/e = {1 / Math.E:F4})");
}

n=   20  P(meilleur selectionne) = 0,3833   (theorie : 1/e = 0,3679)


n=  100  P(meilleur selectionne) = 0,3771   (theorie : 1/e = 0,3679)


n= 1000  P(meilleur selectionne) = 0,3690   (theorie : 1/e = 0,3679)


**Lecture du résultat.** Sur vingt mille tirages par taille, la règle des 37 % capture le meilleur rang avec une fréquence de **0,3833** à $n = 20$, **0,3771** à $n = 100$ et **0,3690** à $n = 1000$, contre $1/e = 0,3679$ en théorie. Les deux premières valeurs dépassent la limite, la troisième la frôle : la correction de taille finie est positive et s'amortit quand $n$ grandit. Le niveau $1/e$ est donc confirmé, le chiffre exact restant un régime asymptotique plutôt qu'une valeur atteinte à $n = 20$.

L'algorithme récupère le meilleur candidat **un peu plus d'une fois sur trois** sur les trois tailles d'instance, et c'est optimal pour ce modèle. Tout l'enjeu du secrétaire **matroïdal** est de savoir ce qui survit quand on ne peut plus tout accepter : sélectionner plusieurs éléments sous une contrainte de structure, avec la même arrivée aléatoire irréversible.

## 2. Le secrétaire matroïdal — la conjecture tombée

**Le problème** : les éléments arrivent en ordre aléatoire, chacun porte une valeur positive ; à chaque arrivée, on décide **immédiatement et irréversiblement** de l'accepter ou non ; l'ensemble accepté doit rester **indépendant** pour un matroïde connu uniquement via un **oracle d'indépendance** (et le nombre total d'éléments $n$). Objectif : maximiser l'espérance de la somme des valeurs acceptées, comparée à l'optimum **offline** — le poids maximum d'un ensemble indépendant, que le glouton sur valeurs triées calcule exactement.

**La conjecture** (Babaioff–Immorlica–Kleinberg 2007) : il existe un algorithme à **compétitivité constante** — un $\alpha > 0$ indépendant de $n$ et du matroïde tel que $\mathbb{E}[\text{valeur acceptée}] \geq \alpha \cdot \mathrm{OPT}$.

**Ce qu'annonce le preprint de Singla** : un algorithme online qui **accepte chaque élément de l'optimum offline avec probabilité au moins $1/4$** — compétitivité constante, alors que l'algorithme ne connaît **pas le matroïde à l'avance** (il n'a que l'oracle d'indépendance et $n$). L'énoncé est dans l'abstract (p. 1) et le théorème 1.1 (p. 2) du PDF archivé. Les meilleurs ratios connus restaient en $O(\log\log r)$, où $r$ est le rang du matroïde (Lachish 2014 ; preuve simplifiée par Feldman–Svensson–Zenklusen 2015) — état de l'art rappelé p. 2.

Côté C#, un matroïde se représente par une interface à une méthode — l'oracle — et le matroïde graphique par un union-find avec compression de chemin, exactement comme la version Python. L'interface est ce qui rend les deux stratégies génériques : elles ne voient jamais la structure, seulement la question « cet ensemble est-il indépendant ? ».

In [3]:
// --- Deux matroides, chacun expose Independant(int[]) -> bool ---

interface IMatroide { bool Independant(int[] S); }

class MatroideUniforme : IMatroide
{
    public readonly int N, K;
    public MatroideUniforme(int n, int k) { N = n; K = k; }
    public bool Independant(int[] S) => S.Length <= K;   // rang k : au plus k elements
}

class MatroideGraphique : IMatroide
{
    readonly (int u, int v)[] aretes;
    public MatroideGraphique((int u, int v)[] aretes) { this.aretes = aretes; }

    // Independants = les forets : un cycle des que deux extremites ont la meme racine.
    public bool Independant(int[] S)
    {
        var parent = new Dictionary<int, int>();
        int Find(int x) { while (parent[x] != x) { parent[x] = parent[parent[x]]; x = parent[x]; } return x; }
        foreach (int i in S)
        {
            var (u, v) = aretes[i];
            if (!parent.ContainsKey(u)) parent[u] = u;
            if (!parent.ContainsKey(v)) parent[v] = v;
            int ru = Find(u), rv = Find(v);
            if (ru == rv) return false;   // arete fermant un cycle : rejetee
            parent[ru] = rv;
        }
        return true;
    }
}

// Sur un matroide, le glouton sur valeurs decroissantes est optimal :
// il rend le poids maximum d'un ensemble independant.
(double poids, int[] acceptes) OptOffline(IMatroide matroide, double[] valeurs)
{
    var acceptes = new List<int>();
    foreach (int i in Enumerable.Range(0, valeurs.Length).OrderByDescending(i => valeurs[i]))
    {
        if (matroide.Independant(acceptes.Append(i).ToArray())) acceptes.Add(i);
    }
    return (acceptes.Sum(i => valeurs[i]), acceptes.ToArray());
}

// Temoins qui montrent que les deux oracles repondent dans les deux sens.
var mu0 = new MatroideUniforme(12, 3);
Console.WriteLine($"uniforme (rang 3) : {{0,1,2}} -> {mu0.Independant(new[] { 0, 1, 2 })}, {{0,1,2,3}} -> {mu0.Independant(new[] { 0, 1, 2, 3 })}");
var triangle = new MatroideGraphique(new[] { (0, 1), (1, 2), (2, 0) });
Console.WriteLine($"graphique (triangle) : {{0,1}} -> {triangle.Independant(new[] { 0, 1 })}, {{0,1,2}} -> {triangle.Independant(new[] { 0, 1, 2 })}");

uniforme (rang 3) : {0,1,2} -> True, {0,1,2,3} -> False


graphique (triangle) : {0,1} -> True, {0,1,2} -> False


### Deux stratégies de référence, aucune n'étant celle de Singla

Avant de comparer quoi que ce soit à la conjecture, il faut deux stratégies online **implémentables en quelques lignes** et honnêtes : le **glouton pur** — accepter tout ce qui reste indépendant, dans l'ordre d'arrivée — et le **glouton à seuil** — observer la première moitié des arrivées, en tirer une valeur médiane, puis n'accepter que ce qui la dépasse. Aucune ne connaît le matroïde à l'avance : elles n'ont que l'oracle d'indépendance, exactement l'interface que le théorème de Singla suppose. C'est ce qui rend la comparaison légitime, et c'est aussi ce qui la borne : ces deux heuristiques n'ont aucune garantie dans le pire cas, elles servent de témoins pour mesurer le dilemme que le théorème résout.

In [4]:
// --- Deux strategies online de reference (ni l'une ni l'autre n'est l'algorithme de Singla) ---

double OnlineGlouton(IMatroide matroide, double[] valeurs, Random r)
{
    var ordre = Permutation(valeurs.Length, r);
    var acceptes = new List<int>();
    foreach (int idx in ordre)
        if (matroide.Independant(acceptes.Append(idx).ToArray())) acceptes.Add(idx);   // accepte tout, tout de suite
    return acceptes.Sum(i => valeurs[i]);
}

double OnlineSeuilMediane(IMatroide matroide, double[] valeurs, Random r, double gamma = 0.5)
{
    var ordre = Permutation(valeurs.Length, r);
    int tObs = (int)(gamma * valeurs.Length);
    var observees = ordre.Take(tObs).Select(i => valeurs[i]).OrderBy(x => x).ToArray();
    double mediane = observees.Length == 0
        ? 0.0
        : (observees.Length % 2 == 1
            ? observees[observees.Length / 2]
            : 0.5 * (observees[observees.Length / 2 - 1] + observees[observees.Length / 2]));
    var acceptes = new List<int>();
    foreach (int idx in ordre.Skip(tObs))
        if (valeurs[idx] >= mediane && matroide.Independant(acceptes.Append(idx).ToArray())) acceptes.Add(idx);
    return acceptes.Sum(i => valeurs[i]);
}

// Le parametre optionnel gamma empeche la conversion du groupe de methodes en Func<> :
// on fige la strategie dans un delegat a trois arguments, reutilise partout ensuite.
Func<IMatroide, double[], Random, double> StratSeuil = (matroide, valeurs, r) => OnlineSeuilMediane(matroide, valeurs, r);

// E[gain] / OPT. OPT ne depend ni du tirage ni de la strategie : memes valeurs,
// meme matroide. Le calculer une fois hors de la boucle ne change aucune mesure
// et evite vingt fois le meme tri.
double Competitivite(IMatroide matroide, double[] valeurs,
                     Func<IMatroide, double[], Random, double> strategie, int runs, Random r)
{
    double opt = OptOffline(matroide, valeurs).poids;
    double somme = 0;
    for (int k = 0; k < runs; k++) somme += strategie(matroide, valeurs, r);
    return somme / runs / opt;
}

### Ce que l'on mesure, et sur quels deux matroïdes

Trois grandeurs par matroïde et par stratégie : l'optimum hors-ligne, calculé une fois — ni le tirage ni la stratégie ne le changent, seules les valeurs et le matroïde comptent — puis le rapport $\mathbb{E}[\text{gain}]/\mathrm{OPT}$ sur deux mille suites tirées au sort. Deux matroïdes de natures opposées servent de banc d'essai. L'**uniforme**, de rang 3 sur 12 éléments, où le rang sature vite et où attendre ne coûte presque rien : observer la première moitié ne consomme aucune capacité. Le **graphique**, sur la grille 3 × 4, où chaque arête écartée peut être précisément celle qui ferme un maillage et où l'attente se paie. C'est ce contraste que la conjecture de Singla doit absorber avec un seul algorithme, sans aucune connaissance préalable du matroïde.

In [5]:
// --- Mesures : matroide uniforme (rang 3 sur 12) puis graphique (grille 3x4) ---

var mu = new MatroideUniforme(12, 3);
var valU = Enumerable.Range(0, 12).Select(_ => 1.0 + 9.0 * rng.NextDouble()).ToArray();

var grille = (from r in Enumerable.Range(0, 3) from c in Enumerable.Range(0, 4) select (r, c)).ToArray();
var aretes = new List<(int, int)>();
for (int r = 0; r < 3; r++)
    for (int c = 0; c < 4; c++)
    {
        int id = r * 4 + c;
        if (c < 3) aretes.Add((id, id + 1));
        if (r < 2) aretes.Add((id, id + 4));
    }
var mg = new MatroideGraphique(aretes.ToArray());
var valG = Enumerable.Range(0, aretes.Count).Select(_ => 1.0 + 9.0 * rng.NextDouble()).ToArray();

Console.WriteLine($"Matroide uniforme (rang 3 / 12 elements), OPT = {OptOffline(mu, valU).poids:F2}");
Console.WriteLine($"  glouton pur      : competitivite = {Competitivite(mu, valU, OnlineGlouton, 2000, rng):F4}");
Console.WriteLine($"  glouton a seuil  : competitivite = {Competitivite(mu, valU, StratSeuil, 2000, rng):F4}");
Console.WriteLine();
Console.WriteLine($"Matroide graphique (grille 3x4, {aretes.Count} aretes), OPT = {OptOffline(mg, valG).poids:F2}");
Console.WriteLine($"  glouton pur      : competitivite = {Competitivite(mg, valG, OnlineGlouton, 2000, rng):F4}");
Console.WriteLine($"  glouton a seuil  : competitivite = {Competitivite(mg, valG, StratSeuil, 2000, rng):F4}");

Matroide uniforme (rang 3 / 12 elements), OPT = 25,95


  glouton pur      : competitivite = 0,6948


  glouton a seuil  : competitivite = 0,7265


Matroide graphique (grille 3x4, 17 aretes), OPT = 79,05


  glouton pur      : competitivite = 0,7664


  glouton a seuil  : competitivite = 0,4391


### Une dispersion, pas un point

Un ratio de compétitivité mesuré sur un seul flux est une estimation, pas un résultat. La cellule suivante rejoue donc la comparaison sur les **huit graines du protocole** déjà employé par la version Python — 0, 1, 7, 42, 99, 123, 256, 1024 — avec deux mille suites par graine, et rapporte l'intervalle, la moyenne et l'écart-type des deux stratégies ainsi que le nombre de graines où chacune passe devant. Un classement qui ne tiendrait qu'à une graine n'aurait aucune valeur ; c'est la dispersion qui décide.

In [6]:
// --- Dispersion du §2 : huit graines independantes sur le matroide uniforme ---
// Le point affiche plus haut sort d'un seul tirage ; ici la comparaison est rejouee
// sur les huit graines du protocole Python (0, 1, 7, 42, 99, 123, 256, 1024) pour que
// la conclusion « aucun reglage ne domine » porte sa propre dispersion.
// Le generateur partage n'est pas touche : les mesures en aval (C_5, recapitulatif)
// restent celles du flux seme 42.
var seeds = new[] { 0, 1, 7, 42, 99, 123, 256, 1024 };
var pur = new List<double>();
var seuil = new List<double>();
foreach (int s in seeds)
{
    var rs = new Random(s);
    pur.Add(Competitivite(mu, valU, OnlineGlouton, 2000, rs));
    seuil.Add(Competitivite(mu, valU, StratSeuil, 2000, rs));
}

double Moy(List<double> v) => v.Average();
double EcartType(List<double> v) { double m = v.Average(); return Math.Sqrt(v.Sum(x => (x - m) * (x - m)) / v.Count); }
int Devant(List<double> a, List<double> b) => a.Zip(b, (x, y) => x > y ? 1 : 0).Sum();

Console.WriteLine($"glouton pur     : {pur.Min():F4} a {pur.Max():F4}  (moy {Moy(pur):F4}, sigma {EcartType(pur):F4})");
Console.WriteLine($"glouton a seuil : {seuil.Min():F4} a {seuil.Max():F4}  (moy {Moy(seuil):F4}, sigma {EcartType(seuil):F4})");
Console.WriteLine($"seuil devant sur {Devant(seuil, pur)}/{seeds.Length} graines ; glouton pur devant sur {Devant(pur, seuil)}/{seeds.Length}");

glouton pur     : 0,6946 a 0,7061  (moy 0,7001, sigma 0,0034)


glouton a seuil : 0,7123 a 0,7427  (moy 0,7274, sigma 0,0080)


seuil devant sur 8/8 graines ; glouton pur devant sur 0/8


**Lecture des mesures.** Sur le matroïde uniforme (rang 3 sur 12 éléments, OPT = 25,95), le glouton pur atteint une compétitivité de **0,6948** et le glouton à seuil **0,7265** : écarter d'abord la moitié des arrivées paie. Sur le matroïde graphique (grille 3 × 4, 17 arêtes, OPT = 79,05) l'ordre s'inverse brutalement — **0,7664** pour le glouton pur contre **0,4391** pour le seuil. La raison est structurelle : écarter les premières arêtes d'une grille, c'est perdre celles qui ferment les maillages et qu'aucune arête tardive ne remplacera. La dispersion sur les huit graines du protocole confirme que le premier classement n'est pas un accident d'échantillonnage — glouton pur de **0,6946** à **0,7061** (moyenne 0,7001, écart-type 0,0034), seuil de **0,7123** à **0,7427** (moyenne 0,7274, écart-type 0,0080), le seuil devant sur **huit graines sur huit**. Cette dispersion porte sur l'uniforme seul : l'inversion graphique n'est mesurée que sur le tirage unique de la cellule précédente, et se lit donc comme un signal, pas comme une statistique.

**Aucune des deux stratégies n'est l'algorithme de Singla.** Le résultat annoncé par le preprint (abstract p. 1, théorème 1.1 p. 2) — accepter chaque élément de l'optimum avec probabilité $\geq 1/4$, **sans connaître le matroïde à l'avance** — est d'une autre nature : il garantit la constante dans le pire cas, pour tout matroïde, avec pour seule interface l'oracle d'indépendance. Ces simulations mesurent le *dilemme* que le théorème résout : entre accepter tôt (structure) et attendre (valeur), aucune heuristique simple ne domine sur tous les matroïdes.

## 3. k-server — le work function algorithm, implémenté

**Le problème** (Manasse–McGeoch–Sleator–Tarjan ~1990) : $k$ serveurs occupent des points d'un espace métrique ; une **séquence de requêtes** (adversariale) arrive, chaque requête est un point ; à chaque requête, un serveur doit s'y déplacer, au coût de la distance parcourue. Objectif : minimiser le coût total, comparé au coût de l'optimum offline qui connaît toute la séquence. Trois quantités à ne pas confondre. **Borne inférieure prouvée** (Manasse–McGeoch–Sleator 1990) : aucun algorithme déterministe n'atteint un ratio inférieur à $k$ sur les métriques à plus de $k$ points — $k$ est le plancher, atteint déjà pour $k = 2$ et $n = k+1$ (rappelé p. 1 du PDF archivé de Coester et al.). **Meilleure garantie prouvée avant 2026** : le **work function algorithm** (WFA) est $(2k-1)$-compétitif sur tout espace métrique (Koutsoupias–Papadimitriou 1995 ; rappelé p. 1). **La conjecture** : le WFA atteint exactement le plancher — ratio $k$ sur **tout** espace métrique — c'est ce que le preprint de Coester–Koutsoupias–Zbysiński annonce (abstract p. 1 ; théorème 1.2 p. 3).

**Le WFA**, de l'aveu général, est « l'algorithme évident qu'on n'arrivait pas à analyser » : à chaque requête, maintenir la **work function** $W_t(S)$ = coût minimal pour avoir servi les $t$ premières requêtes et se trouver en configuration $S$ ; servir la requête courante par le mouvement qui minimise *(coût immédiat + work function résultante)* — un regard à la fois vers le passé optimal et vers le futur.

Nous l'implémentons **exactement** sur deux petites métriques — la **ligne** uniforme à $m$ points et le **cycle** $C_m$ où l'on peut passer par le bord — où la work function se calcule par programmation dynamique sur les $\binom{m}{k}$ configurations, pour $k = 2$ serveurs. En C#, une configuration est un couple trié d'entiers et la table des work functions un tableau de dictionnaires : la recombinaison se lit sans index, et une configuration finale qui ne contient pas le point requis est simplement absente de la table.

In [7]:
// --- Work function algorithm exact pour k = 2, metrique parametree (ligne / cycle) ---

Func<int, int, double> MakeLigne(int m) => (i, j) => Math.Abs(i - j);

// Cycle C_m : d(i, j) = min(|i-j|, m-|i-j|) -- on peut passer par le bord.
Func<int, int, double> MakeCycle(int m) => (i, j) => Math.Min(Math.Abs(i - j), m - Math.Abs(i - j));

(int, int)[] Configs(int m, int k = 2) =>
    (from a in Enumerable.Range(0, m) from b in Enumerable.Range(a + 1, m - a - 1) select (a, b)).ToArray();

// Cout minimal pour passer de la config s1 a la config s2 (assignment k = 2 : deux affectations).
double Transition((int, int) s1, (int, int) s2, Func<int, int, double> d)
{
    if (s1 == s2) return 0.0;
    var (a, b) = s1;
    var (c, e) = s2;
    return Math.Min(d(a, c) + d(b, e), d(a, e) + d(b, c));
}

// W[t][S] = cout minimal pour servir requetes[0..t-1] et finir en S (S contient requetes[t-1]).
Dictionary<(int, int), double>[] WorkFunctions(int m, (int, int) init, int[] requetes, Func<int, int, double> d, int k = 2)
{
    var cfg = Configs(m, k);
    var W = new Dictionary<(int, int), double>[requetes.Length + 1];
    W[0] = cfg.ToDictionary(S => S, S => Transition(init, S, d));   // cout initial de couverture
    for (int t = 1; t <= requetes.Length; t++)
    {
        int r = requetes[t - 1];
        W[t] = new Dictionary<(int, int), double>();
        foreach (var S in cfg)
        {
            if (S.Item1 != r && S.Item2 != r) continue;   // une config finale doit contenir la requete
            double best = double.PositiveInfinity;
            foreach (var kv in W[t - 1]) best = Math.Min(best, kv.Value + Transition(kv.Key, S, d));
            W[t][S] = best;
        }
    }
    return W;
}

// Simule la trajectoire WFA : a chaque requete, le mouvement minimisant (cout + W_t(resultat)).
(double cout, double opt) JouerWfa(int m, (int, int) init, int[] requetes, Func<int, int, double> d, int k = 2)
{
    var W = WorkFunctions(m, init, requetes, d, k);
    var S = (Math.Min(init.Item1, init.Item2), Math.Max(init.Item1, init.Item2));
    double coutTotal = 0.0;
    for (int t = 1; t <= requetes.Length; t++)
    {
        int r = requetes[t - 1];
        var candidats = new List<(int, int)>();
        if (S.Item1 == r || S.Item2 == r) candidats.Add(S);        // un serveur est deja sur la requete
        else
        {
            candidats.Add((Math.Min(S.Item1, r), Math.Max(S.Item1, r)));
            candidats.Add((Math.Min(S.Item2, r), Math.Max(S.Item2, r)));
        }
        (int, int) best = S;
        double bestCout = double.PositiveInfinity;
        foreach (var cand in candidats)
        {
            double total = Transition(S, cand, d) + W[t][cand];
            if (total < bestCout) { best = cand; bestCout = total; }
        }
        coutTotal += Transition(S, best, d);   // le cout REEL paye, pas le cout estime
        S = best;
    }
    double opt = W[requetes.Length].Values.Min();
    return (coutTotal, opt);
}

// Baseline : bouge toujours le serveur le plus proche de la requete.
double JouerGreedy(int m, (int, int) init, int[] requetes, Func<int, int, double> d)
{
    var servs = new[] { init.Item1, init.Item2 };
    double coutTotal = 0.0;
    for (int t = 0; t < requetes.Length; t++)
    {
        int r = requetes[t];
        int i = Enumerable.Range(0, 2).OrderBy(j => d(servs[j], r)).First();
        coutTotal += d(servs[i], r);
        servs[i] = r;
    }
    return coutTotal;
}

### Ce que l'on mesure, et pourquoi le cycle

Deux expériences, choisies pour être **vérifiables à la main**. D'abord le piège canonique du **cycle $C_5$**, les deux serveurs du même côté : l'optimum réorganise une fois puis sert tout gratuitement, tandis que le glouton « serveur le plus proche » suit l'alternance des requêtes et paie à chaque pas — un coût qui croît avec la longueur de la séquence, donc un ratio qui **diverge**. Ensuite une **ligne à huit points** sous une séquence aléatoire, c'est-à-dire le régime moyen, où le glouton redevient présentable. Le contraste est tout l'intérêt : la borne $k$ du WFA se lit au pire cas, jamais en moyenne.

In [8]:
// --- Mesures : le piege canonique du CYCLE C_5, puis la ligne en contraste ---
// Cycle a 5 points, serveurs en (0, 1) -- tous deux du meme cote -- requetes alternant 3, 4.
// OPT paie une reorganisation (0 -> 4 par le bord a cout 1, 1 -> 3 a cout 2) puis sert tout a 0.
// Le greedy, lui, suit l'alternance : a chaque requete le serveur le plus proche bascule de 3 a 4.

var dC5 = MakeCycle(5);
const int k = 2;
int[] Alternance(int paires) => Enumerable.Range(0, paires).SelectMany(_ => new[] { 3, 4 }).ToArray();

var (cWfa, opt) = JouerWfa(5, (0, 1), Alternance(30), dC5);
var cGr = JouerGreedy(5, (0, 1), Alternance(30), dC5);
Console.WriteLine("CYCLE C_5, init (0,1), requetes [3,4] x 30 (T=60) :");
Console.WriteLine($"  OPT offline        = {opt:F0}");
Console.WriteLine($"  WFA                = {cWfa:F0}   ratio = {cWfa / opt:F3}  (borne k = {k})");
Console.WriteLine($"  greedy plus-proche = {cGr:F0}   ratio = {cGr / opt:F1}");
Console.WriteLine();
foreach (int paires in new[] { 60, 240 })
{
    var (cw, o) = JouerWfa(5, (0, 1), Alternance(paires), dC5);
    double cg = JouerGreedy(5, (0, 1), Alternance(paires), dC5);
    Console.WriteLine($"  T={paires * 2,4} requetes : ratio WFA = {cw / o:F3}   ratio greedy = {cg / o:F1}  (greedy croit avec T)");
}
Console.WriteLine();

var dL8 = MakeLigne(8);
var alea = Enumerable.Range(0, 60).Select(_ => rng.Next(8)).ToArray();
var (cw2, o2) = JouerWfa(8, (0, 7), alea, dL8);
double cg2 = JouerGreedy(8, (0, 7), alea, dL8);
Console.WriteLine("LIGNE a 8 points, sequence aleatoire (60 requetes) -- le regime moyen est facile :");
Console.WriteLine($"  OPT offline = {o2:F0}   ratio WFA = {cw2 / o2:F3}   ratio greedy = {cg2 / o2:F3}");

CYCLE C_5, init (0,1), requetes [3,4] x 30 (T=60) :


  OPT offline        = 3


  WFA                = 5   ratio = 1,667  (borne k = 2)


  greedy plus-proche = 61   ratio = 20,3


  T= 120 requetes : ratio WFA = 1,667   ratio greedy = 40,3  (greedy croit avec T)


  T= 480 requetes : ratio WFA = 1,667   ratio greedy = 160,3  (greedy croit avec T)


LIGNE a 8 points, sequence aleatoire (60 requetes) -- le regime moyen est facile :


  OPT offline = 79   ratio WFA = 1,025   ratio greedy = 1,101


**Lecture des mesures.** Sur le cycle $C_5$, l'optimum hors-ligne paie **3** : réorganiser les deux serveurs (0 → 4 par le bord, coût 1 ; 1 → 3, coût 2) puis servir toute la séquence gratuitement. Le WFA paie **5**, soit un ratio de **1,667** — sous la borne $k = 2$, et **stable quand la séquence s'allonge** : 1,667 encore à cent vingt puis quatre cent quatre-vingts requêtes. Le glouton « serveur le plus proche », lui, paie **61** (ratio **20,3**) et son coût croît d'un cran à chaque requête ajoutée : 40,3 puis 160,3 aux deux longueurs suivantes, c'est-à-dire un ratio qui **diverge** avec $T$. La ligne à huit points sur une séquence aléatoire est le contraste : OPT = 79, WFA à **1,025**, glouton à **1,101** — un régime moyen où tous deux s'en sortent, et où le WFA reste légèrement devant. Ces trois ratio 1,667 sont exactement ceux de la version Python : l'expérience du $C_5$ ne dépend d'aucun tirage, seules les valeurs tirées du reste du carnet diffèrent entre les deux carnets.

Ce que le preprint de Coester–Koutsoupias–Zbysiński annonce (abstract p. 1, théorème 1.2 p. 3) : le WFA atteint le ratio $k$ **sur tout espace métrique et toute séquence** — la technique (représentation algébrique de la work function, chaque valeur comme déterminant de $k$ colonnes, potentiel sur paires de coordonnées) est décrite p. 1, 3 et 4 du PDF archivé, citée §5, pas reproduite ici.

## 4. Le double visage du binôme

Ce qui rend la saisie **commune** de ces deux résultats nécessaire — c'est le même problème (décisions irréversibles sous incertitude, évaluées au ratio compétitif) résolu par deux philosophies exactement opposées :

| | Secrétaire matroïdal (Singla) | k-server (Coester–Koutsoupias–Zbysiński) |
|---|---|---|
| Modèle d'entrée | ordre **aléatoire** (hypothèse distributionnelle) | séquence **adversariale** (aucune hypothèse) |
| Arme | **randomisation** (accepte chaque élément de l'OPT avec proba $\geq 1/4$, p. 1–2) | **déterminisme** (le WFA, un algorithme unique, p. 1) |
| Analyse | espérance, probabilités | analyse amortie, fonction potentiel |
| Conjecture ouverte depuis | 2007 (dix-neuf ans) | ~1990 (trente-six ans) |
| Outil structurel | matroïdes (oracle d'indépendance, p. 1) | algèbre des work functions (déterminants, p. 3) |
| Ce que la machine mesure | compétitivité Monte Carlo $\approx$ constante | ratio sur un adversaire construit, sous la borne $k$ (le pire cas est l'affirmation du théorème 1.2, p. 3) |

Le jumeau C# ajoute une ligne à ce tableau, invisible dans la version Python : **l'outillage**. Les deux modèles se simulent avec la seule BCL — un `Random` semé pour l'arrivée aléatoire, un `Dictionary` de couples pour la work function — sans tableau numérique ni dépendance externe. Ce que le carnet perd au passage, c'est la reproductibilité croisée avec Python : mêmes graines, générateurs différents, donc mesures propres à chaque version. Les deux carnets s'accordent sur ce qui ne dépend pas du générateur — la borne $k$, le ratio du WFA sur l'adversaire construit, la croissance linéaire du greedy.

In [9]:
// Recapitulatif chiffre des mesures de ce carnet
var resume = new (string mesure, string valeur, string reference)[]
{
    ("Secretaire classique n=1000 : P(meilleur)", $"{MonteCarloClassique(1000, 5000, rng):F4}", "1/e = 0.3679"),
    ("Matroide uniforme : glouton pur", $"{Competitivite(mu, valU, OnlineGlouton, 2000, rng):F4}", "aucune garantie generale (le 1/4 est celui de Singla)"),
    ("Matroide uniforme : glouton a seuil", $"{Competitivite(mu, valU, StratSeuil, 2000, rng):F4}", "aucune garantie generale (seuil optimal selon l'instance)"),
    ("k-server cycle C5 : ratio WFA", $"{cWfa / opt:F3}", $"borne k = {k}"),
    ("k-server cycle C5 : ratio greedy", $"{cGr / opt:F1} (croit avec T)", "non competitif (2k-1 est la garantie du WFA, KP95)"),
};
Console.WriteLine($"{"Mesure",-46}{"valeur",10}   reference");
foreach (var (mesure, valeur, reference) in resume)
    Console.WriteLine($"{mesure,-46}{valeur,10}   {reference}");

Mesure                                            valeur   reference


Secretaire classique n=1000 : P(meilleur)         0,3738   1/e = 0.3679


Matroide uniforme : glouton pur                   0,6982   aucune garantie generale (le 1/4 est celui de Singla)


Matroide uniforme : glouton a seuil               0,7283   aucune garantie generale (seuil optimal selon l'instance)


k-server cycle C5 : ratio WFA                      1,667   borne k = 2


k-server cycle C5 : ratio greedy              20,3 (croit avec T)   non competitif (2k-1 est la garantie du WFA, KP95)


## 5. Prudence épistémique — ce que ce carnet cite sans le vérifier

Les deux papiers sont des **preprints de un à deux jours** au moment de la rédaction (soumis les 13 et 14/09/2026), sans évaluation par les pairs publiée, sans écho communautaire enregistré. Les formulations de ce carnet disent donc **« le preprint annonce »**, **« la preuve annoncée »** — jamais « le théorème » ni « résolu » sans qualificatif. C'est la même discipline que pour la semaine discrepancy : la saisie pédagogique peut être immédiate, la canonisation ne peut pas l'être.

Trois niveaux à ne pas confondre, dans ce carnet :

1. **Mesuré** : les probabilités et ratios produits par les simulations de §1–§3, reproductibles à graine et à version de runtime constantes.
2. **Cité** : les énoncés des deux preprints (compétitivité $1/4$ sans connaissance du matroïde ; ratio $k$ du WFA sur tout espace métrique) — des **claims d'auteurs**, lus sur les abstracts et les énoncés de théorèmes, avec la page du PDF archivé.
3. **Absent** : les preuves elles-mêmes — non lues ligne à ligne ici, non formalisées, non vérifiées mécaniquement.

Si l'une des deux preuves s'effondre en évaluation, ce carnet garde sa valeur pédagogique (les objets, le dilemme, les implémentations de référence) et perdra seulement ses deux phrases d'ouverture — il est conçu pour ça.

## 6. Encart Mathlib — ce qui existe, ce qui manque

Même exercice que pour [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) : où est la frontière de Mathlib (v4.33.0 au jour de rédaction) sur ces objets ?

- **Matroïdes** : présents — `Mathlib.Combinatorics.Matroid` couvre indépendance, rang, dualité, matroïdes graphiques. Le vocabulaire du §2 est formalisable **dès aujourd'hui** (définir « un ensemble est indépendant » et l'algorithme glouton offline y trouverait son cadre).
- **Algorithmique online** : absente — pas de `competitive ratio`, pas de `secretary`, pas de `k-server`, pas de `work function`, pas de notion d'algorithme online face à une séquence de requêtes. Formaliser la compétitivité exigerait une couche « adversaire + algorithme + ratio sur toutes les entrées » qui n'existe pas plus que la couche `TIME(f)` du carnet 03.
- Le pont naturel si un jour ce grain passe en Lean : la work function est **décidable** (une DP finie) — le calcul est implémentable, c'est la **preuve de ratio** (le potentiel sur paires de coordonnées du preprint) qui serait le travail dur.

La suite logique côté dépôt serait un `online_lean` quand les preuves auront mûri — pas avant.

## 7. Exercices

Trois exercices, stubs exécutables sans erreur (règle C.1 du dépôt) — les corrigés restent la propriété de l'étudiante et de l'étudiant. Chaque cellule porte un **contrôle automatique** qui compare la réponse à une référence écrite dans la cellule : il refuse un stub vide, accepte une réponse juste, et rejette une réponse fausse (un mutant) au lieu de la valider par défaut.

### Exercice 1 — Matroïde de partition (prolonge la section 2)

**Contexte.** La section 2 a comparé deux stratégies en ligne sur deux matroïdes, chacun exposant un oracle `Independant(int[])`. Le matroïde de partition est un troisième cas : chaque élément porte une couleur parmi trois groupes, et un ensemble est indépendant s'il contient au plus $q_i$ éléments du groupe $i$.

**Objectif.** Compléter la méthode `MatroidePartition.Independant`, puis mesurer la compétitivité du glouton à seuil sur cette structure avec `Competitivite`, comme dans la cellule de mesure de la section 2.

**Indices.**
- `S` est un tableau d'indices d'éléments : `couleurs[i]` donne la couleur de l'élément `i`, et `quotas` donne le quota de chaque couleur.
- Il suffit de compter, pour chaque couleur, combien d'éléments de `S` la portent, puis de comparer ce compte au quota correspondant.

**Critère de vérification.** Sur un petit jeu (quatre éléments, deux couleurs, quota 1 pour chacune), `Independant(new[] {0, 1})` doit renvoyer `false` si les éléments 0 et 1 partagent une couleur, et `true` sinon ; `Independant(Array.Empty<int>())` doit renvoyer `true`. Une fois l'oracle complété, `OptOffline` et `Competitivite` doivent s'exécuter sans erreur sur `MatroidePartition`.

In [10]:
// Exercice 1 -- Matroide de partition : chaque element porte une couleur parmi 3 groupes,
// et un ensemble est independant ssi il contient au plus q_i elements de chaque groupe i.
// TODO etudiant : implementer l'oracle puis mesurer la competitivite du glouton a seuil.
// Indice : Independant(S) = toutes les tailles d'intersection groupe_i ∩ S sont <= q_i.

class MatroidePartition : IMatroide
{
    readonly int[] couleurs;                       // couleur de chaque element
    readonly Dictionary<int, int> quotas;          // quota par couleur

    public MatroidePartition(int[] couleurs, Dictionary<int, int> quotas)
    {
        this.couleurs = couleurs;
        this.quotas = quotas;
    }

    public bool Independant(int[] S)
    {
        // TODO etudiant
        return false;
    }
}

// --- Controle automatique : accepte une reponse juste, rejette un mutant ---

// Reference independante de la reponse de l'etudiant (comptage naif par groupe).
bool IndependantReference(int[] couleurs, Dictionary<int, int> quotas, int[] S)
{
    foreach (var groupe in S.GroupBy(i => couleurs[i]))
        if (groupe.Count() > quotas[groupe.Key]) return false;
    return true;
}

// Reponse candidate : on eprouve l'oracle sur toutes les parties d'un jeu de 4 elements,
// de facon a rejeter aussi bien l'oracle toujours vrai que l'oracle toujours faux.
bool ControlePartition(IMatroide oracle, int[] couleurs, Dictionary<int, int> quotas)
{
    if (oracle is null) return false;
    var elements = Enumerable.Range(0, couleurs.Length).ToArray();
    int vus = 0;
    foreach (int masque in Enumerable.Range(0, 1 << elements.Length))
    {
        var S = elements.Where(i => (masque & (1 << i)) != 0).ToArray();
        bool attendu = IndependantReference(couleurs, quotas, S);
        bool obtenu;
        try { obtenu = oracle.Independant(S); } catch { return false; }
        if (obtenu != attendu) return false;
        vus++;
    }
    return vus == (1 << elements.Length);
}

var couleursEx1 = new[] { 0, 0, 1, 1 };
var quotasEx1 = new Dictionary<int, int> { [0] = 1, [1] = 1 };
var partition = new MatroidePartition(couleursEx1, quotasEx1);

if (ControlePartition(partition, couleursEx1, quotasEx1))
{
    var valP = Enumerable.Range(0, 12).Select(_ => 1.0 + 9.0 * rng.NextDouble()).ToArray();
    var couleurs12 = Enumerable.Range(0, 12).Select(i => i % 3).ToArray();
    var quotas12 = new Dictionary<int, int> { [0] = 2, [1] = 2, [2] = 2 };
    var mp = new MatroidePartition(couleurs12, quotas12);
    Console.WriteLine($"Oracle valide sur les 16 parties du jeu temoin ; OPT = {OptOffline(mp, valP).poids:F2} ; competitivite du glouton a seuil = {Competitivite(mp, valP, StratSeuil, 2000, rng):F4}");
}
else
{
    Console.WriteLine("Exercice a completer : compter, par couleur, les elements de S et comparer au quota.");
}


// --- Preuve du garde : il accepte une solution juste et rejette un mutant ---
// La reference ci-dessous n'est pas une reponse a recopier : elle fixe ce que le
// controle doit accepter, pour qu'un eleve puisse verifier qu'il peut le passer.

// Une classe d'une soumission .NET Interactive ne voit pas les fonctions locales du
// script (CS0120) : l'adaptateur ci-dessous transporte la fonction, le lambda etant
// cree dans la portee du script.
class MatroideFonction : IMatroide
{
    readonly Func<int[], bool> f;
    public MatroideFonction(Func<int[], bool> f) { this.f = f; }
    public bool Independant(int[] S) => f(S);
}

var oracleRef = new MatroideFonction(S => IndependantReference(couleursEx1, quotasEx1, S));
var oracleToutVrai = new MatroideFonction(S => true);   // mutant : accepte tout, donc surestime

Console.WriteLine($"garde ex.1 -- oracle de reference accepte : {ControlePartition(oracleRef, couleursEx1, quotasEx1)} (attendu True)");
Console.WriteLine($"garde ex.1 -- mutant 'tout est independant' rejete : {!ControlePartition(oracleToutVrai, couleursEx1, quotasEx1)} (attendu True)");

Exercice a completer : compter, par couleur, les elements de S et comparer au quota.


garde ex.1 -- oracle de reference accepte : True (attendu True)


garde ex.1 -- mutant 'tout est independant' rejete : True (attendu True)


### Exercice 2 — k-server sur l'étoile (prolonge la section 3)

**Contexte.** La section 3 a mesuré le work function algorithm et le glouton sur le cycle $C_5$, où le glouton suit l'alternance des requêtes. L'étoile à $m$ feuilles est un autre espace métrique pour deux serveurs : un point central $0$ et des feuilles $1, \dots, m$, avec $d(0, i) = 1$ et $d(i, j) = 2$ pour deux feuilles distinctes. **Convention de comptage des points** : l'étoile à $m$ feuilles en compte $m+1$ (le centre plus les feuilles), alors que `MakeCycle(m)` et `MakeLigne(m)` en décrivent $m$ — c'est donc $m+1$ qu'il faut passer comme nombre de points à `JouerWfa` et `JouerGreedy`.

**Objectif.** Compléter `MakeEtoile(m)`, qui doit renvoyer la fonction de distance au même format que `MakeCycle` et `MakeLigne` (en substituant les $m+1$ points de l'étoile). Puis chercher une séquence de requêtes qui fait payer cher le glouton (`JouerGreedy`), et vérifier avec `JouerWfa` que le work function algorithm reste sous la borne $k = 2$ (en passant bien $m+1$ comme nombre de points).

**Indices.**
- `MakeCycle` et `MakeLigne` renvoient une `Func<int, int, double>` : `MakeEtoile` doit faire de même.
- Le contrôle porte sur $m = 3$ ; la mesure qui suit porte sur $m = 4$ feuilles, donc cinq points. Un adversaire efficace place les deux serveurs sur deux feuilles et requête alternativement deux **autres** feuilles : le glouton bascule à chaque requête, alors qu'un seul déplacement initial de chaque serveur suffit à l'optimum hors-ligne. Comparez le coût total de `JouerGreedy` et de `JouerWfa` à la valeur `opt` renvoyée par ce dernier.

**Critère de vérification.** `MakeEtoile(3)(0, 2)` doit valoir `1`, `MakeEtoile(3)(1, 2)` doit valoir `2`, et la fonction doit être nulle sur la diagonale. Sur la séquence trouvée, le rapport `coût WFA / opt` doit rester inférieur ou égal à $k = 2$.

In [11]:
// Exercice 2 -- k-server sur l'ETOILE a m feuilles : point 0 = le centre,
// points 1..m = les feuilles ; d(i, j) = 1 si l'un est le centre, 2 entre
// deux feuilles distinctes.
// TODO etudiant : implementer MakeEtoile(m) puis chercher une sequence adversariale
// pour le greedy et verifier que le WFA reste sous la borne k = 2.

Func<int, int, double> MakeEtoile(int m)
{
    // TODO etudiant : renvoyer une fonction d(i, j) sur les m+1 points de l'etoile.
    return null;
}

// --- Controle automatique : distances attendues + rejet d'une metrique constante ---

bool ControleEtoile(Func<int, int, double> d)
{
    if (d == null) return false;
    bool ok = true;
    for (int i = 0; i <= 3; i++) ok &= Math.Abs(d(i, i)) < 1e-12;          // diagonale nulle
    ok &= Math.Abs(d(0, 2) - 1) < 1e-12;                                    // centre -> feuille
    ok &= Math.Abs(d(1, 2) - 2) < 1e-12;                                    // feuille -> feuille
    ok &= Math.Abs(d(1, 2) - d(2, 1)) < 1e-12;                              // symetrie
    for (int i = 0; i <= 3; i++)
        for (int j = 0; j <= 3; j++)
            ok &= d(i, j) <= d(i, 0) + d(0, j) + 1e-12;                     // inegalite triangulaire
    return ok;
}

var etoile3 = MakeEtoile(3);
if (ControleEtoile(etoile3))
{
    // Mesure sur m = 4 feuilles (5 points). Serveurs sur les feuilles 1 et 2,
    // requetes alternant les feuilles 3 et 4 : le greedy bascule a chaque pas
    // (2 a chaque requete), l'optimum paie deux deplacements puis sert gratuitement.
    var dEtoile = MakeEtoile(4);
    var requetesEtoile = Enumerable.Range(0, 30).SelectMany(_ => new[] { 3, 4 }).ToArray();
    var (cEtoile, oEtoile) = JouerWfa(5, (1, 2), requetesEtoile, dEtoile);
    double gEtoile = JouerGreedy(5, (1, 2), requetesEtoile, dEtoile);
    Console.WriteLine($"Metrique d'etoile valide ; OPT = {oEtoile:F0}, WFA = {cEtoile:F0} (ratio {cEtoile / oEtoile:F3}), greedy = {gEtoile:F0} (ratio {gEtoile / oEtoile:F1})");
    Console.WriteLine($"Borne k = {k} respectee par le WFA : {cEtoile / oEtoile <= k}");
}
else
{
    Console.WriteLine("Exercice a completer : distance 1 depuis le centre, 2 entre feuilles, 0 sur la diagonale.");
}


// --- Preuve du garde : la metrique juste est acceptee, une metrique constante rejetee ---
Func<int, int, double> EtoileReference(int m) => (i, j) => i == j ? 0.0 : (i == 0 || j == 0 ? 1.0 : 2.0);
Func<int, int, double> EtoileConstante(int m) => (i, j) => 1.0;   // mutant : tout le monde a distance 1

Console.WriteLine($"garde ex.2 -- metrique de reference acceptee : {ControleEtoile(EtoileReference(4))} (attendu True)");
Console.WriteLine($"garde ex.2 -- mutant 'toute distance vaut 1' rejete : {!ControleEtoile(EtoileConstante(4))} (attendu True)");

Exercice a completer : distance 1 depuis le centre, 2 entre feuilles, 0 sur la diagonale.


garde ex.2 -- metrique de reference acceptee : True (attendu True)


garde ex.2 -- mutant 'toute distance vaut 1' rejete : True (attendu True)


### Exercice 3 — Secrétaire à valeurs log-normales (prolonge la section 1)

**Contexte.** La section 1 a mesuré la règle du $1/e$ sur des permutations de rangs. Ici on reprend la même stratégie sur des valeurs réelles tirées d'une loi log-normale, dont la queue est lourde : quelques valeurs très grandes dominent la moyenne.

**Objectif.** Compléter `SecretaireLognormal(n, mu, sigma, rng)` : tirer $n$ valeurs log-normales, rejeter les $n/e$ premières, accepter le premier record relatif, et renvoyer `true` si le meilleur candidat a été capturé. Estimer ensuite la probabilité de succès sur un grand nombre d'essais, et la comparer à $1/e$.

**Indices.**
- La stratégie ne dépend que de l'ordre relatif des valeurs : la logique de `SecretaireClassique` s'applique telle quelle, lue sur les valeurs au lieu des rangs.
- Une loi log-normale se tire par `Math.Exp(mu + sigma * z)` avec $z$ gaussien — `Math.Sqrt(-2 * Math.Log(u)) * Math.Cos(2 * Math.PI * v)` (Box–Muller) en fournit un à partir de deux uniformes.

**Critère de vérification.** `SecretaireLognormal` renvoie un booléen ; sur plusieurs milliers d'essais avec $n = 100$, la fréquence de succès doit rester voisine de $1/e \approx 0{,}368$, à l'écart d'échantillonnage près.

In [12]:
// Exercice 3 -- Secretaire classique avec valeurs log-normales (queue lourde) au lieu de
// valeurs uniformes : montrer que la strategie de RANG (rejeter les n/e premiers observes,
// puis accepter le premier record relatif) reste valable quelle que soit la loi des valeurs.
// TODO etudiant : tirer n valeurs log-normales, appliquer la regle, renvoyer le succes.

bool? SecretaireLognormal(int n, Random r, double mu = 0.0, double sigma = 1.0)
{
    // TODO etudiant : utiliser Permutation pour l'ordre d'arrivee, ou trier les valeurs tirees.
    return null;
}

// --- Controle automatique : un booleen n'est pas une mesure ; on exige la bonne frequence ---

bool ControleLognormal(Func<int, Random, double, double, bool?> strategie)
{
    var temoin = new Random(7);
    int runs = 4000, succes = 0;
    for (int i = 0; i < runs; i++)
    {
        bool? rep;
        try { rep = strategie(100, temoin, 0.0, 1.0); } catch { return false; }
        if (rep is null) return false;              // stub non complete
        if (rep.Value) succes++;
    }
    double p = (double)succes / runs;
    // Bande d'echantillonnage large : 4000 essais, ecart-type ~0,0076, on tolere +/- 0,04.
    return Math.Abs(p - 1 / Math.E) < 0.04;
}

if (ControleLognormal(SecretaireLognormal))
{
    var rl = new Random(123);
    int runs = 20_000, ok = 0;
    for (int i = 0; i < runs; i++) if (SecretaireLognormal(100, rl, 0.0, 1.0) == true) ok++;
    Console.WriteLine($"Secretaire log-normal n=100 : P(meilleur) = {(double)ok / runs:F4}   (theorie : 1/e = {1 / Math.E:F4})");
}
else
{
    Console.WriteLine("Exercice a completer : la frequence de succes doit approcher 1/e = 0,3679 sur n = 100.");
}


// --- Preuve du garde : la strategie de rang est acceptee, le mutant sans observation rejete ---

// Les deux essais ci-dessous partagent le tirage : les rangs d'un vecteur de valeurs
// i.i.d. continues sont uniformement distribues, c'est tout l'argument de l'exercice.
int[] RangsLognormaux(int n, Random r, double mu, double sigma)
{
    var v = new double[n];
    for (int i = 0; i < n; i++)
    {
        double u1 = 1.0 - r.NextDouble(), u2 = r.NextDouble();          // u1 dans (0, 1]
        double z = Math.Sqrt(-2.0 * Math.Log(u1)) * Math.Cos(2.0 * Math.PI * u2);   // Box-Muller
        v[i] = Math.Exp(mu + sigma * z);
    }
    var perm = new int[n];
    var ordre = Enumerable.Range(0, n).OrderBy(i => v[i]).ToArray();     // du plus petit au plus grand
    for (int rang = 0; rang < n; rang++) perm[ordre[rang]] = rang;
    return perm;
}

bool? SecretaireLognormalReference(int n, Random r, double mu, double sigma)
    => SecretaireClassique(n, RangsLognormaux(n, r, mu, sigma));

bool? SecretaireSansObservation(int n, Random r, double mu, double sigma)
{
    var perm = RangsLognormaux(n, r, mu, sigma);
    return perm[0] == n - 1;   // mutant : accepte le premier arrive, sans fenetre d'observation
}

Console.WriteLine($"garde ex.3 -- strategie de rang acceptee : {ControleLognormal(SecretaireLognormalReference)} (attendu True)");
Console.WriteLine($"garde ex.3 -- mutant 'sans observation' rejete : {!ControleLognormal(SecretaireSansObservation)} (attendu True)");

Exercice a completer : la frequence de succes doit approcher 1/e = 0,3679 sur n = 100.


garde ex.3 -- strategie de rang acceptee : True (attendu True)


garde ex.3 -- mutant 'sans observation' rejete : True (attendu True)


## 8. Conclusion

Ce carnet a **saisi le domaine online côté Complexity**, en C# et sans dépendance externe : la baseline $1/e$ mesurée (§1), le dilemme du secrétaire matroïdal simulé sur deux matroïdes avec oracles d'indépendance (§2), le work function algorithm **implémenté exactement** et confronté à ses bornes sur séquence adversariale (§3), le tableau du double visage randomisation/déterminisme (§4). Les deux preprints sont cités au conditionnel de rigueur (§5), avec renvoi paginé, le gap Mathlib est cartographié (§6), et trois exercices prolongent chacun une section (§7), chacun muni d'un contrôle automatique qui rejette un mutant.

**Ce que le portage C# déplace, et ce qu'il ne déplace pas.** Les budgets, les graines et les expériences sont ceux de la version Python ; le générateur, lui, est celui de la BCL. Les grandeurs qui ne dépendent pas du hasard sont identiques dans les deux carnets — l'optimum offline sur le cycle $C_5$, le coût du work function algorithm sur l'adversaire construit, la croissance linéaire du glouton. Les valeurs tirées, elles, diffèrent dans les décimales et les deux carnets l'affichent plutôt que de le taire.

**Cross-links** : [Complexity-03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) (même méthode : faire tourner, confronter à Mathlib) · [`Complexity-04c-KServer-WorkFunction-Python.ipynb`](Complexity-04c-KServer-WorkFunction-Python.ipynb) (k-server : WFA exact sur métriques jouets — même preprint, en profondeur) · [`Complexity-04d-Secretaire-Matroidal-Python.ipynb`](Complexity-04d-Secretaire-Matroidal-Python.ipynb) (l'algorithme de Singla implémenté, garantie 1/4 comme banc de mesure) · [`RL/RL-16-Dream-RSI-Python.ipynb`](../../RL/RL-16-Dream-RSI-Python.ipynb) (l'exploration explicite programmable, l'autre bout du spectre online).

## À retenir

- **Secrétaire classique** : la règle des 37 % mesure 0,3833 / 0,3771 / 0,3690 pour $n$ = 20 / 100 / 1000 ; la limite $1/e = 0,3679$ est confirmée, approchée par au-dessus aux petites tailles.
- **Secrétaire matroïdal** : sur l'uniforme le seuil (0,7265) bat le glouton pur (0,6948), et huit graines confirment ce classement ; sur le graphique l'inverse est net (0,4391 contre 0,7664). Aucun réglage ne domine les deux structures — c'est le dilemme que la garantie $1/4$ de Singla tranche.
- **k-server** : le WFA paie 5 contre un OPT de 3 sur $C_5$ (ratio 1,667, sous la borne $k = 2$) à toute longueur ; le glouton paie 61 (ratio 20,3) et son ratio croît sans borne avec la longueur.
- **Portage** : générateur de la BCL au lieu de PCG64 — les grandeurs sans hasard (3, 5, 1,667) sont identiques au carnet Python ; les grandeurs tirées en diffèrent, et les deux carnets l'affichent.

## Vérifiez votre compréhension

1. La probabilité mesurée dépasse $1/e$ sur les petites instances : contradiction avec l'optimalité de la règle des 37 % ?
2. L'inversion entre matroïdes — le seuil devant sur l'uniforme, derrière sur le graphique — que faut-il en conclure pour le choix d'un algorithme ?
3. Le glouton fait un ratio énorme sur $C_5$ et un ratio proche de 1 sur la ligne aléatoire. Que garantit exactement le $(2k-1)$ du WFA, et que ne garantit-il pas ?

<details><summary>Réponses</summary>

1. Non : les petites instances portent des corrections de taille finie, et l'écart s'amortit sans disparaître à mesure que $n$ grandit. Le niveau $1/e$ est confirmé ; le chiffre exact est un régime asymptotique, pas une valeur atteinte à $n = 20$.
2. Qu'aucun réglage n'est sûrement meilleur : la compétitivité dépend de la structure d'indépendance. C'est le dilemme du secrétaire matroïdal ; la garantie $1/4$ de Singla (implémentée dans 04d) est la réponse qui ne dépend pas de l'instance.
3. Un pire cas sur **toute** séquence : le WFA reste sous $k$ quelle que soit la longueur, ce que la mesure confirme en rejouant l'adversaire à plusieurs longueurs. Le glouton n'est pas compétitif — son ratio croît sans borne avec $T$ — et il n'est sauvé que par des suites faciles comme la ligne aléatoire, ce qui n'est pas une garantie.

</details>

## Pour aller plus loin

- **Version Python du même carnet** : [Complexity-04b — Secrétaire matroïdal et k-server](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) — mêmes expériences, générateur PCG64, et les chiffres de référence de la série.
- **Carnet suivant** : [Complexity-04c — k-server : work function](Complexity-04c-KServer-WorkFunction-Python.ipynb) — le WFA exact en profondeur, sur le même preprint Coester–Koutsoupias–Zbysiński.
- **Dans la même série** : [Complexity-04d — Secrétaire matroïdal](Complexity-04d-Secretaire-Matroidal-Python.ipynb) — l'algorithme de Singla implémenté, garantie 1/4 comme banc de mesure ; [Complexity-04 — Algorithmes en ligne (C#)](Complexity-04-OnlineAlgorithms-CSharp.ipynb) — les trois problèmes de base.
- **Carnet voisin d'une autre série** : [RL-16 — Dream-RSI (Python)](../../RL/RL-16-Dream-RSI-Python.ipynb) — cité par la Conclusion : l'exploration explicite programmable, l'autre bout du spectre en ligne.
- **Lectures** : les deux preprints (k-server, secrétaire matroïdal) sont cités au conditionnel de rigueur en §5.